In [ ]:
import sys
sys.path.append("..")

import numpy as np
import wfdb
from sklearn.metrics import classification_report, f1_score
from src.pipeline import load_model, analyze, analyze_portable, CLASSES, AAMI_MAP

bundle = load_model("../models/ecg_rf_small.joblib")
DATA_DIR = "../data/mitdb"
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210,
       212, 213, 214, 219, 221, 222, 228, 231, 232, 233, 234]


def evaluate(fn, name):
    y_true, y_pred, offs = [], [], []
    n_ann = n_det = n_tp = 0

    for rec in DS2:
        record = wfdb.rdrecord(f"{DATA_DIR}/{rec}")
        ann = wfdb.rdann(f"{DATA_DIR}/{rec}", "atr")
        ch = record.sig_name.index("MLII")
        res = fn(record.p_signal[:, ch], record.fs, bundle)

        keep = [i for i, s in enumerate(ann.symbol) if s in AAMI_MAP]
        a_samples = ann.sample[keep]
        a_labels = [AAMI_MAP[ann.symbol[i]] for i in keep]

        n_ann += len(a_samples)
        n_det += len(res["peaks"])
        for p, c in zip(res["peaks"], res["pred"]):
            j = np.argmin(np.abs(a_samples - p))
            if abs(a_samples[j] - p) <= 36:          # at most 0.1 s away
                n_tp += 1
                offs.append(a_samples[j] - p)
                if a_labels[j] in CLASSES:
                    y_true.append(CLASSES.index(a_labels[j]))
                    y_pred.append(c)

    print("=" * 20, name)
    print("annotated:", n_ann, "| detected:", n_det, "| matched:", n_tp)
    print("sensitivity:", round(n_tp / n_ann, 4),
          "| positive predictivity:", round(n_tp / n_det, 4))
    print("offset percentiles 5/50/95:", np.percentile(offs, [5, 50, 95]))
    print(classification_report(y_true, y_pred, labels=[0, 1, 2, 3],
                                target_names=CLASSES, digits=3, zero_division=0))
    print("macro F1 (N,S,V):",
          round(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro"), 3))


evaluate(analyze_portable, "final pipeline: portable detector + 50-tree forest")

==================== XQRS + scipy filter (current)
annotated: 49712 | detected: 49559 | matched: 49498
sensitivity: 0.9957 | positive predictivity: 0.9988
offset percentiles 5/50/95: [-1.  0.  1.]
              precision    recall  f1-score   support

           N      0.954     0.970     0.962     44192
           S      0.176     0.157     0.166      1824
           V      0.953     0.903     0.927      3092
           F      0.000     0.000     0.000       384

    accuracy                          0.928     49492
   macro avg      0.521     0.508     0.514     49492
weighted avg      0.918     0.928     0.923     49492

macro F1 (N,S,V): 0.685
==================== portable filter + portable detector
annotated: 49712 | detected: 49680 | matched: 49564
sensitivity: 0.997 | positive predictivity: 0.9977
offset percentiles 5/50/95: [-1.  0.  1.]
              precision    recall  f1-score   support

           N      0.956     0.970     0.963     44167
           S      0.176     0.155